# E2B 回答长度与跨块诊断
从原始 AR E2B 开始，不续接算术 checkpoint-900。第一阶段拟合 49 token 固定回答；通过后保留它并加入 293 token 回答，跨过 C=256。长度含原始结束标记，运行时再用真实分词器检查。
保持 C=256、K容量=18、全画布平均 CE、BF16 AdamW、LR=1.125e-4、前50步预热。每阶段最多400步，每100步评测4个固定种子；连续两次整段全对才晋级，失败停止，无全量训练。K是最大容量，不是每个样本必须18块。
同时报告整段自由生成的第一/第二块、给定正确前文的独立块生成、各块正文/EOS尾部恢复率。正确前文测试只是诊断，不替代整段验收。
先选 A100 80GB。完整包为 llamafactory_length_controls_v1.tgz。Drive建议预留100GiB以上，检查点包含优化器并保留最近两份；Drive网页配额请另外确认。真实E2B时长尚未实测。

In [ ]:
import torch
assert torch.cuda.is_available(), '先连接 A100 80GB'
free, total = torch.cuda.mem_get_info()
print(torch.cuda.get_device_name(), '总/空闲GiB', total/1024**3, free/1024**3)
assert total >= 75*1024**3 and free >= 45*1024**3, '显存不足，尚未加载模型或开始训练'


In [ ]:
from google.colab import files, userdata, drive
from pathlib import Path
import os, sys, tarfile, subprocess
files.upload()  # 选择 llamafactory_length_controls_v1.tgz
with tarfile.open('/content/llamafactory_length_controls_v1.tgz') as archive:
    archive.extractall('/content', filter='data')
os.environ['HF_TOKEN'] = userdata.get('HF')
os.environ['PYTHONPATH'] = '/content/src'
os.environ['DISABLE_VERSION_CHECK'] = '1'
os.environ['WANDB_DISABLED'] = 'true'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==5.23.0', 'safetensors', 'sentencepiece', 'pytest', 'psutil'], check=True)
drive.mount('/content/drive')
OUT='/content/drive/MyDrive/saves/e2b_length_controls_v1'
sys.path.insert(0, '/content/scripts')
from run_single_sample_controls import preflight, fresh_destination
fresh_destination(Path(OUT))
print(preflight(out=OUT))


In [ ]:
tests=sorted(str(p) for p in Path('/content/tests/train').glob('test_diffusion_*.py'))
tests += ['/content/tests/train/test_single_sample_controls.py', '/content/tests/train/test_length_controls.py']
r=subprocess.run([sys.executable, '-m', 'pytest', '-q', *tests, '--confcutdir=/content/tests/train'], capture_output=True, text=True)
print(r.stdout); print(r.stderr); r.check_returncode()


In [ ]:
cmd=[sys.executable, '-u', '/content/scripts/run_single_sample_controls.py', '--suite', 'length', '--out', OUT]
with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
    for line in process.stdout:
        print(line, end='', flush=True)
    code=process.wait()
if code != 0:
    raise RuntimeError(f'退出码{code}，实际错误已显示。不要重复开训。')


In [ ]:
r=subprocess.run([sys.executable, '/content/scripts/run_single_sample_controls.py', '--suite', 'length', '--out', OUT, '--summary-only'], capture_output=True, text=True)
print(r.stdout); print(r.stderr); r.check_returncode()


## 中断恢复
只有含COMPLETE的检查点可恢复。使用相同代码和参数，新建空输出目录，再增加 --resume <旧检查点目录>；不要覆盖旧实验。latest_checkpoint.json提供最近保存路径。恢复包含优化器与阶段/RNG状态。
EVAL 的 stage_size=1 表示只训练49-token回答；stage_size=2表示短回答和293-token回答一起训练，不是C或K发生改变。FREE_BLOCKS的block=0/1是自由生成第一/第二块；CORRECT_PREFIX_BLOCK表示给了正确前文的独立块测试。其通过不能替代整段自由生成。